In [1]:
import sys
sys.path.insert(0, '../'); del sys

In [2]:
import tensorflow as tf

/usr/local/lib/python3.5/dist-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters


In [3]:
from models import Encoder, Decoder, BahdanauAttention
from setting import BATCH_SIZE, UNITS, EMBEDDING_DIM, ATTENTION_UNITS, MODELPATH
from get_data import en_tok, kr_tok, en_seq_train, kr_seq_train, en_wc, kr_wc

# make encoder
vocab_inp_size = len(en_tok.word_index) + 1
vocab_tar_size = len(kr_tok.word_index) + 1
encoder = Encoder(vocab_inp_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)
decoder = Decoder(vocab_tar_size, EMBEDDING_DIM, UNITS, BATCH_SIZE)

In [4]:
import matplotlib.pyplot as plt
_ = plt.hist(en_wc, bins=50, range=(0, 100))
plt.ylim(0, 100)

(0, 100)

In [5]:
checkpoint = tf.train.Checkpoint(encoder=encoder, decoder=decoder)

In [6]:
checkpoint.restore(tf.train.latest_checkpoint(MODELPATH))

In [14]:
import numpy as np
from input_data import preprocess_eng
from get_data import en_tok, kr_tok

inp_lang = en_tok
targ_lang = kr_tok

def max_length(tensor):
    return max(len(t) for t in tensor)

max_length_inp = max_length(en_seq_train)
max_length_targ = max_length(kr_seq_train)

def evaluate(sentence):
    attention_plot = np.zeros((max_length_targ, max_length_inp))

    sentence = preprocess_eng(sentence)

    #inputs = [inp_lang.word_index[i] for i in sentence.split(' ')]
    inputs = []
    for i in sentence.split(' '):
        try:
            inputs.append(inp_lang.word_index[i])
        except KeyError:
            inputs.append(inp_lang.word_index['<UNK>'])
    inputs = tf.keras.preprocessing.sequence.pad_sequences([inputs],
                                                           maxlen=max_length_inp,
                                                           padding='post')
    inputs = tf.convert_to_tensor(inputs)

    result = ''

    hidden = [tf.zeros((1, UNITS))]
    enc_out, enc_hidden = encoder(inputs, hidden)

    dec_hidden = enc_hidden
    dec_input = tf.expand_dims([targ_lang.word_index['<start>']], 0)

    for t in range(max_length_targ):
        predictions, dec_hidden, attention_weights = decoder(dec_input,
                                                             dec_hidden,
                                                             enc_out)

        # storing the attention weights to plot later on
        attention_weights = tf.reshape(attention_weights, (-1, ))
        attention_plot[t] = attention_weights.numpy()

        predicted_id = tf.argmax(predictions[0]).numpy()

        result += targ_lang.index_word[predicted_id] + ' '

        if targ_lang.index_word[predicted_id] == '<end>':
            return result, sentence, attention_plot

        # the predicted ID is fed back into the model
        dec_input = tf.expand_dims([predicted_id], 0)

    return result, sentence, attention_plot

def translate(sentence):
    result, sentence, attention_plot = evaluate(sentence)

    print('Input: %s' % (sentence))
    print('Predicted translation: {}'.format(result))

    attention_plot = attention_plot[:len(result.split(' ')), :len(sentence.split(' '))]
    plot_attention(attention_plot, sentence.split(' '), result.split(' '))

In [22]:
sent = 'i don\'t lie'
#sent = 'i love youd'
tgt, src, c = evaluate(sent)

In [23]:
'{} → {}'.format(tgt, src)

'나는 거짓말 하지 않아 <end>  → <start> i don t lie <end>'